# Capítulo 9 — Otimização de inferência: rápido e barato

## Missão
Medir e cortar **latência e custo** do assistente. Você vai medir as métricas que o usuário sente (tempo até o primeiro token, tempo por token), ver percentis, ganhar vazão com chamadas em paralelo e criar **cache exato e semântico**. Também vai fazer as contas que explicam o hardware: memória do KV cache, limite por memória × por cálculo e decodificação especulativa.

**No fim deste caderno você terá:** o módulo `meu_assistente/cache.py` (cache exato + semântico), que o assistente final usa para responder perguntas repetidas sem chamar o modelo.

**Leitura essencial (opcional, livro em inglês):**
- Inference Overview · PDF p. 430
- Inference Performance Metrics · PDF p. 436
- Model Optimization · PDF p. 450
- Inference Service Optimization · PDF p. 464

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap09_otimizacao_de_inferencia_rapido_e_barato.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, re
sys.path.insert(0, "../comum")                 # ia.py (acesso aos modelos) e dados.py (dados da Nimbus)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_assistente/
os.makedirs("../meu_assistente", exist_ok=True)
open("../meu_assistente/__init__.py", "a").close()        # meu_assistente/ é o projeto que você constrói ao longo do curso
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
import numpy as np
import pandas as pd
from ia import *
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Modelo:", MODELO, "| modo simulado:", FAKE, "| seu pacote: meu_assistente")

## Ideia em 30 segundos
Gerar texto tem duas fases:
- **prefill**: ler o prompt inteiro; é limitado pelo cálculo e define o tempo até o primeiro token, o **TTFT**;
- **decode**: gerar token por token; é limitado pela memória e define o tempo por token, o **TPOT**.

Para o usuário, *streaming* faz a resposta parecer mais rápida. Para o caixa, o que conta é a **vazão** (requisições por segundo) e o que você deixa de chamar (cache).

### 9.1 **TTFT e TPOT com streaming.** Com `conversar(..., stream=True)`, peça uma explicação de 3 frases. Marque o tempo do primeiro pedaço de texto (TTFT) e o total. Calcule o TPOT aproximado = (total - TTFT) / (número de pedaços - 1).

### 9.2 **Percentis.** A média esconde os piores casos. Faça 8 chamadas curtas, guarde as latências e mostre p50, p90 e o máximo.

### 9.3 **Vazão com concorrência.** Responda 8 perguntas de `PERGUNTAS`: primeiro em sequência, depois com `ThreadPoolExecutor(max_workers=4)`. Compare o tempo total e as requisições por segundo. (Chamadas de API passam a maior parte do tempo esperando a rede, por isso threads ajudam.)

### 9.4 **Limite a saída.** Tokens de saída custam mais e demoram mais (são gerados um a um). Compare a mesma pergunta com `max_tokens=300` e com `max_tokens=40` mais a instrução "responda em uma frase": latência e tokens de saída.

### 9.5 **Cache exato.** Escreva `perguntar_com_cache(prompt)`, que guarda a resposta num dict com a chave `sha256(modelo + prompt)`. Chame duas vezes a mesma pergunta e mostre o tempo de cada chamada.

### 9.6 **Cache semântico.** Perguntas iguais escritas de jeitos diferentes deveriam reaproveitar a resposta. Guarde (vetor, resposta) numa lista. Numa pergunta nova, se a maior similaridade com as guardadas passar de um **limiar**, devolva a guardada. Teste com 3 paráfrases de "Quanto custa o plano Equipe?" e uma pergunta diferente, com limiar 0,85. Que risco um limiar baixo demais traz?

### 9.7 **Módulo de cache.** Escreva `cache.py` com a classe `CacheRespostas(limiar=0.9)`:
- `buscar(pergunta)` primeiro tenta o cache **exato** (texto normalizado) e depois o **semântico**; devolve `(resposta, tipo)` ou `(None, None)`;
- `guardar(pergunta, resposta)`;
- `estatisticas()`, com acertos exatos, semânticos e faltas.

_A célula grava `meu_assistente/cache.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_assistente/cache.py


### 9.8 **Teste o cache.** Crie `CacheRespostas(limiar=0.9)`, guarde 2 respostas e busque: a mesma pergunta, uma variação de maiúsculas/espaços e uma pergunta nova. Mostre as estatísticas.

### 9.9 **Memória do KV cache.** Para cada token, cada camada guarda K e V: `2 × camadas × cabeças_kv × dim_cabeça × bytes`. Para um modelo tipo 8B (32 camadas, 8 cabeças KV, dimensão 128, fp16), calcule a memória do KV cache para contexto de 8.192 tokens e lotes de 1, 16 e 64 requisições.

### 9.10 **Limitado por memória ou por cálculo?** Uma GPU de exemplo faz 312 TFLOP/s e lê 2 TB/s da memória; o ponto de virada é `312e12 / 2e12` FLOPs por byte. No *decode* com lote 1 de um modelo de 8B em fp16, cada token faz ~`2 × 8e9` FLOPs e lê ~`8e9 × 2` bytes. Calcule a intensidade aritmética para lotes de 1 a 256 e veja a partir de qual lote o decode deixa de ser limitado por memória. É por isso que servidores juntam requisições em lotes.

### 9.11 **Decodificação especulativa.** Um modelo pequeno propõe `k` tokens e o grande confere todos de uma vez. Se cada token é aceito com probabilidade `α`, o número esperado de tokens por passo do modelo grande é `(1 - α^(k+1)) / (1 - α)`. Monte a tabela para α = 0,6, 0,8 e 0,9 e k = 2, 4 e 8. Depois **simule** α=0,8, k=4 com sorteios e confira a fórmula.

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero testar com o modelo de verdade (fora do modo simulado):
